In [1]:
import numpy as np

from engin_core.gp import fit_gp, split_conformal_multiplier
from engin_core.simulator import Kinetics, simulate_unit

NOMINAL, D, TRAIN_HI = 0.90, 5, 0.6
SEEDS = range(5)

def observed(U, rng, kinetics=None):
    """Titer with heteroscedastic measurement noise."""
    y = simulate_unit(U, kinetics=kinetics)
    return np.maximum(y + rng.normal(0, 0.05 * y + 0.4), 0.0)

def calibrated_model(seed):
    """Fit and conformally calibrate on the lower 60% of every design axis."""
    rng = np.random.default_rng(seed)
    Utr = rng.uniform(0, TRAIN_HI, (70, D))
    Uca = rng.uniform(0, TRAIN_HI, (30, D))
    gp = fit_gp(Utr, observed(Utr, rng), seed=seed)
    mc, sdc = gp.predict(Uca, include_noise=True)
    q = split_conformal_multiplier(observed(Uca, rng), mc, sdc, level=NOMINAL)
    return gp, q, rng

In [2]:
REGIONS = [
    ("in-distribution", 0.0, 0.6),
    ("just outside",    0.6, 0.7),
    ("far outside",     0.9, 1.0),
]

results = {name: [] for name, _, _ in REGIONS}
for seed in SEEDS:
    gp, q, rng = calibrated_model(seed)
    for name, lo, hi in REGIONS:
        U = rng.uniform(lo, hi, (40, D))
        y = observed(U, rng)
        m, sd = gp.predict(U, include_noise=True)
        err = np.abs(m - y)
        results[name].append((np.mean(err <= q * sd), np.mean(2 * q * sd), np.mean(err)))

print(f"  {'region':<17}{'coverage':>9}{'width':>9}{'error':>9}")
for name, vals in results.items():
    cov, width, mae = np.array(vals).mean(axis=0)
    print(f"  {name:<17}{cov:>9.3f}{width:>9.1f}{mae:>9.1f}")

  region            coverage    width    error
  in-distribution      0.930      9.3      1.7
  just outside         0.870     18.9      3.7
  far outside          0.965     76.2     13.0


In [3]:
VARIANTS = {
    "same process":                  Kinetics(),
    "stronger inhibition kp 18->6":  Kinetics(kp=6.0),
    "slower growth mu_max .35->.22": Kinetics(mu_max=0.22),
    "several at once":               Kinetics(kp=8.0, alpha=0.05, mu_max=0.26),
}

print(f"  {'test process':<32}{'coverage':>9}")
for label, kin in VARIANTS.items():
    covs = []
    for seed in SEEDS:
        gp, q, rng = calibrated_model(seed)
        U = rng.uniform(0, TRAIN_HI, (40, D))
        y = observed(U, rng, kinetics=kin)
        m, sd = gp.predict(U, include_noise=True)
        covs.append(np.mean(np.abs(m - y) <= q * sd))
    print(f"  {label:<32}{np.mean(covs):>9.3f}")

  test process                     coverage


  same process                        0.930


  stronger inhibition kp 18->6        0.935


  slower growth mu_max .35->.22       0.705


  several at once                     0.700
